# Studio Framework Tests — LLM-only Mode

Same stack prerequisites as `studio-framework-hybrid.ipynb`, but all evaluations
run with `--mode llm-only` (retrieval bypassed, Ollama-hosted `primus-reasoning`
answers from its own weights).

**Cells (top → bottom):**
1. Prereq (env + MPS + stack up + deep smoke)
2. Single test-id — `B03-001`
3. Single benchmark — `B3` (30 cases, ~20–30 min)
4. Full suite — all 18 benchmarks, 435 cases (~5+ h)

Cells 3 and 4 are expensive. Run them one at a time when you're ready —
the prereq cell is idempotent, so no need to re-run it before each.

**Kernel:** `studio-ssdlc (poetry)`.


In [1]:
# =====================================================================
# Prerequisites: device=mps + stack startup + health check
# =====================================================================
import os
import subprocess
import sys
from pathlib import Path

import torch
from dotenv import load_dotenv

# -- 0. Paths -----------------------------------------------------------
REPO_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip())
SRC_DIR = REPO_ROOT / "src"
ENV_FILE = SRC_DIR / "config" / ".env.local"
START_SCRIPT = SRC_DIR / "scripts" / "start_local.sh"

print(f"Repo root:   {REPO_ROOT}")
print(f"Env file:    {ENV_FILE}  (exists={ENV_FILE.exists()})")
print(f"Startup:     {START_SCRIPT}  (exists={START_SCRIPT.exists()})")
print(f"Python:      {sys.executable}")
print()

# -- 1. Load .env.local -------------------------------------------------
loaded = load_dotenv(ENV_FILE, override=True)
print(f"[env]   Loaded {ENV_FILE.name}: {loaded}")

# -- 2. Device = mps ----------------------------------------------------
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    torch.set_default_device("mps")
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    print(f"[torch] default device = mps  (torch {torch.__version__}, fallback=1)")
else:
    DEVICE = torch.device("cpu")
    print(f"[torch] MPS unavailable, falling back to cpu  (torch {torch.__version__})")

_sanity = (torch.randn(4, 4, device=DEVICE) @ torch.randn(4, 4, device=DEVICE)).sum().item()
print(f"[torch] device sanity matmul: {_sanity:.4f}")
print()

# Streaming runner: streams stdout (stderr merged) in real time, returns buffered result.
from collections import namedtuple as _namedtuple

RunResult = _namedtuple("RunResult", ["stdout", "returncode"])

def run_script(args, timeout=None):
    proc = subprocess.Popen(
        [str(START_SCRIPT), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    buf = []
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
        proc.wait(timeout=timeout)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait(timeout=10)
        raise
    return RunResult(stdout="".join(buf), returncode=proc.returncode)


# -- 3. Ensure stack is up ---------------------------------------------
print("[stack] checking status...")
status = run_script(["--status"])
if status.returncode != 0:
    print("\n[stack] degraded - running startup (first-run ingestion can take several minutes)...\n")
    startup = run_script([], timeout=1800)
    if startup.returncode != 0:
        raise RuntimeError(f"start_local.sh failed with exit code {startup.returncode}")
else:
    print("[stack] already healthy - skipping startup")
print()

# -- 4. Health check (deep smoke) --------------------------------------
print("[health] running deep smoke test...")
health = run_script(["--status", "--deep"], timeout=300)
if health.returncode != 0:
    raise RuntimeError(f"health check FAILED (exit {health.returncode}) - see output above")
print("[health] PASS - stack is healthy, ready for tests")


Repo root:   /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc
Env file:    /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/config/.env.local  (exists=True)
Startup:     /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/scripts/start_local.sh  (exists=True)
Python:      /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/.venv/bin/python

[env]   Loaded .env.local: True
[torch] default device = mps  (torch 2.7.1, fallback=1)
[torch] device sanity matmul: 1.3619

[stack] checking status...
── CCoP Local Environment Status ──
[OK]    Qdrant: UP at http://localhost:6333
[OK]    Ollama: UP (externally managed)
[OK]    Collection 'ccop_clauses_hybrid': present (points_count=495)
[OK]    Model 'primus-reasoning': loaded
──────────────────────────────────
[OK]    All checks passed
[stack] already healthy - skipping startup

[health] running deep smoke test...
── CCoP Local Environment Status ──
[OK]    Qdrant: UP at http://localhost:6333
[OK]    Ollama: UP (externally mana

## 1. Single test-id — LLM-only mode

- **Test ID:** `B03-001`
- **Mode:** `llm-only` (no retrieval)
- **Phase:** baseline
- **Expected duration:** ~1–2 min


In [3]:
# Single test-id eval in LLM-only mode (streaming output)
TEST_ID = "B03-001"

proc = subprocess.Popen(
    ["poetry", "run", "ccop-eval", "evaluate", "run",
     "--model", "primus-reasoning",
     "--mode", "llm-only",
     "--test-ids", TEST_ID],
    cwd=SRC_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait(timeout=10)
    raise
print(f"\nexit_code = {proc.returncode}")


{"event": "Discovered benchmark file: b01_ccop_applicability_scope.jsonl -> B01", "timestamp": "2026-04-22T23:55:36.354318Z", "level": "info"}
{"event": "Discovered benchmark file: b02_compliance_classification.jsonl -> B02", "timestamp": "2026-04-22T23:55:36.355531Z", "level": "info"}
{"event": "Discovered benchmark file: b03_conditional_compliance_reasoning.jsonl -> B03", "timestamp": "2026-04-22T23:55:36.356262Z", "level": "info"}
{"event": "Discovered benchmark file: b04_it_ot_classification_boundary.jsonl -> B04", "timestamp": "2026-04-22T23:55:36.356729Z", "level": "info"}
{"event": "Discovered benchmark file: b05_control_comprehension.jsonl -> B05", "timestamp": "2026-04-22T23:55:36.357217Z", "level": "info"}
{"event": "Discovered benchmark file: b06_intent_understanding.jsonl -> B06", "timestamp": "2026-04-22T23:55:36.357583Z", "level": "info"}
{"event": "Discovered benchmark file: b07_gap_identification_quality.jsonl -> B07", "timestamp": "2026-04-22T23:55:36.357929Z", "level"

## 2. Single benchmark — LLM-only mode

- **Benchmark:** `B3` (conditional compliance reasoning, 30 cases)
- **Mode:** `llm-only`
- **Expected duration:** ~20–30 min


In [ ]:
# Single benchmark eval in LLM-only mode (streaming output)
BENCHMARK = "B3"

proc = subprocess.Popen(
    ["poetry", "run", "ccop-eval", "evaluate", "run",
     "--model", "primus-reasoning",
     "--mode", "llm-only",
     "--benchmarks", BENCHMARK],
    cwd=SRC_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,     # merge stderr into stdout
    text=True,
    bufsize=1,                    # line-buffered
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait(timeout=10)
    raise
print(f"\nexit_code = {proc.returncode}")


## 3. Full suite — LLM-only mode

All 18 benchmarks, 435 test cases.

- **Mode:** `llm-only`
- **Expected duration:** multiple hours — launch and leave running
- Results land under `src/results/evaluations/` and can be reviewed via
  `poetry run ccop-eval report summary --model primus-reasoning`.


In [ ]:
# Full suite eval in LLM-only mode (streaming output)
proc = subprocess.Popen(
    ["poetry", "run", "ccop-eval", "evaluate", "run",
     "--model", "primus-reasoning",
     "--mode", "llm-only"],
    cwd=SRC_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait(timeout=10)
    raise
print(f"\nexit_code = {proc.returncode}")
